# Notebook 2: Baseline Attacks — FGSM and PGD

**Goal:** Implement and evaluate FGSM and PGD attacks. These are our baselines that the GA attack will be compared against.

**Output:** Attack success rates saved to `../results/tables/baseline_results.csv`

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import sys
sys.path.append('../')

from src.model import CNN
from src.attacks import fgsm_attack, pgd_attack, evaluate_attack

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load trained model
model = CNN().to(device)
model.load_state_dict(torch.load('../results/model.pth', map_location=device))
model.eval()
print('Model loaded.')

In [ ]:
# Load test data (without normalization for visualization)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
testset = torchvision.datasets.CIFAR10(root='../data', train=False, download=False, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

classes = ('plane','car','bird','cat','deer','dog','frog','horse','ship','truck')

In [ ]:
# Visualize adversarial examples
images, labels = next(iter(testloader))
images, labels = images.to(device), labels.to(device)

fgsm_fn = lambda m, x, y: fgsm_attack(m, x, y, epsilon=0.03)
pgd_fn  = lambda m, x, y: pgd_attack(m, x, y, epsilon=0.03)

adv_fgsm = fgsm_fn(model, images, labels)
adv_pgd  = pgd_fn(model, images, labels)

fig, axes = plt.subplots(3, 8, figsize=(16, 6))
titles = ['Original', 'FGSM', 'PGD']
samples = [images, adv_fgsm, adv_pgd]

for row, (title, imgs) in enumerate(zip(titles, samples)):
    for col in range(8):
        img = imgs[col].cpu().detach().permute(1,2,0).numpy()
        img = (img * 0.2 + 0.45).clip(0,1)
        axes[row][col].imshow(img)
        axes[row][col].axis('off')
        if col == 0:
            axes[row][col].set_ylabel(title, fontsize=12)

plt.tight_layout()
plt.savefig('../results/figures/adversarial_examples_baseline.png', dpi=150)
plt.show()

In [ ]:
# Evaluate FGSM and PGD at different epsilon values
epsilons = [0.01, 0.03, 0.05, 0.1]
results = {'epsilon': epsilons, 'fgsm_asr': [], 'pgd_asr': []}

for eps in epsilons:
    fgsm_fn = lambda m, x, y: fgsm_attack(m, x, y, epsilon=eps)
    pgd_fn  = lambda m, x, y: pgd_attack(m, x, y, epsilon=eps)

    fgsm_res = evaluate_attack(model, testloader, fgsm_fn, device, max_samples=500)
    pgd_res  = evaluate_attack(model, testloader, pgd_fn, device, max_samples=500)

    results['fgsm_asr'].append(fgsm_res['attack_success_rate'])
    results['pgd_asr'].append(pgd_res['attack_success_rate'])

    print(f"eps={eps:.2f} | FGSM ASR: {fgsm_res['attack_success_rate']:.3f} | PGD ASR: {pgd_res['attack_success_rate']:.3f}")

In [ ]:
# Plot results
plt.figure(figsize=(8,5))
plt.plot(epsilons, results['fgsm_asr'], 'o-', label='FGSM')
plt.plot(epsilons, results['pgd_asr'],  's-', label='PGD')
plt.xlabel('Epsilon (perturbation budget)')
plt.ylabel('Attack Success Rate')
plt.title('Baseline Attack Performance')
plt.legend()
plt.grid(True)
plt.savefig('../results/figures/baseline_attack_comparison.png', dpi=150)
plt.show()